# Machine Learning Based Burning Feet Syndrome Symptom Assessment

**B.Tech Computer Science — Machine Learning Academic Project**

## 1. Introduction

Burning Feet Syndrome refers to a group of sensations (burning, tingling, numbness) reported in
the feet, often associated with factors such as diabetes, vitamin deficiency, thyroid conditions,
alcohol use, and peripheral neuropathy.

This notebook builds a **multi-class classification** system that takes symptom and health-history
inputs and produces a **preliminary, ML-based assessment category**: `Low`, `Moderate`, or `High`.

> **IMPORTANT DISCLAIMER:** This project uses a **synthetic/demonstration dataset**. It is an
> academic exercise in applied machine learning and does **not** provide a real medical diagnosis.
> Results must not be interpreted as clinically validated.

## 2. Import Libraries

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import joblib

sns.set_style('whitegrid')
RANDOM_STATE = 42
%matplotlib inline

## 3. Load Dataset

If the dataset does not yet exist, generate it first by running `python generate_dataset.py` from the project root.

In [ ]:
df = pd.read_csv('../data/burning_feet_dataset.csv')
df.head()

## 4. Dataset Information

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
df.describe(include='all').T

## 5. Data Cleaning

- Remove duplicate rows
- Impute missing numeric values with the median
- Impute missing categorical values with the mode
- Clip out-of-range values

In [ ]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print('Removed duplicates. New shape:', df_clean.shape)

df_clean['duration_weeks'] = df_clean['duration_weeks'].fillna(df_clean['duration_weeks'].median())
df_clean['severity_level'] = df_clean['severity_level'].fillna(df_clean['severity_level'].median())
df_clean['physical_activity'] = df_clean['physical_activity'].fillna(df_clean['physical_activity'].mode()[0])

df_clean['age'] = df_clean['age'].clip(1, 110)
df_clean['duration_weeks'] = df_clean['duration_weeks'].clip(0, 104)
df_clean['severity_level'] = df_clean['severity_level'].clip(1, 10)

print('Missing values after cleaning:')
print(df_clean.isnull().sum().sum())

## 6. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df_clean, x='assessment', order=['Low','Moderate','High'], palette='viridis', ax=ax)
ax.set_title('Class Distribution of Assessment Category')
ax.set_xlabel('Assessment Category'); ax.set_ylabel('Count')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.histplot(df_clean['age'], bins=20, kde=True, ax=ax, color='steelblue')
ax.set_title('Age Distribution'); ax.set_xlabel('Age'); ax.set_ylabel('Frequency')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(data=df_clean, x='gender', hue='gender', legend=False, palette='pastel', ax=ax)
ax.set_title('Gender Distribution')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, ['burning_sensation', 'tingling', 'numbness']):
    sns.countplot(data=df_clean, x=col, hue=col, legend=False, palette='Set2', ax=ax)
    ax.set_title(f'{col} distribution')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.countplot(data=df_clean, x='assessment', hue='diabetes_history', order=['Low','Moderate','High'], ax=axes[0], palette='coolwarm')
axes[0].set_title('Diabetes History vs Assessment')
sns.countplot(data=df_clean, x='assessment', hue='vitamin_deficiency_history', order=['Low','Moderate','High'], ax=axes[1], palette='coolwarm')
axes[1].set_title('Vitamin Deficiency vs Assessment')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.histplot(df_clean['severity_level'], bins=10, ax=ax, color='orange')
ax.set_title('Severity Level Distribution')
plt.show()

In [ ]:
numeric_df = df_clean.select_dtypes(include=['int64','float64'])
fig, ax = plt.subplots(figsize=(11,8))
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.show()

## 7. Feature Engineering

- **symptom_score**: count of positive symptom indicators (0-8)
- **risk_factor_count**: count of positive health-history indicators (0-5)
- **duration_category**: symptom duration grouped into bins

These summarize signal already present in raw columns without leaking the target.

In [ ]:
binary_symptom_cols = ['burning_sensation','tingling','numbness','foot_pain',
                       'warmth_sensation','redness','swelling','night_time_symptoms']
binary_health_cols = ['diabetes_history','vitamin_deficiency_history','thyroid_condition',
                      'alcohol_risk_factor','peripheral_neuropathy_history']

df_fe = df_clean.copy()
df_fe['symptom_score'] = df_fe[binary_symptom_cols].sum(axis=1)
df_fe['risk_factor_count'] = df_fe[binary_health_cols].sum(axis=1)
df_fe['duration_category'] = pd.cut(df_fe['duration_weeks'],
                                     bins=[-0.1,2,8,26,1000],
                                     labels=['Acute (<2wks)','Short-term (2-8wks)','Chronic (8-26wks)','Long-term (26wks+)']).astype(str)
df_fe.head()

## 8. Preprocessing (Encoding + Scaling via ColumnTransformer)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_cols = ['age','duration_weeks','severity_level','symptom_score','risk_factor_count']
categorical_cols = ['gender','physical_activity','duration_category']
binary_cols = binary_symptom_cols + binary_health_cols

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ('bin', Pipeline([('imputer', SimpleImputer(strategy='most_frequent'))]), binary_cols),
])

feature_cols = numeric_cols + categorical_cols + binary_cols
X = df_fe[feature_cols]
y = df_fe['assessment']
print(X.shape, y.shape)

## 9. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('Train:', X_train.shape, '  Test:', X_test.shape)

## 10. Model Training (Multiple Algorithms)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='accuracy')
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_score': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'cv_mean_accuracy': cv_scores.mean(),
    })
    fitted[name] = pipe
    print(f'{name}: CV Acc={cv_scores.mean():.4f} | Test Acc={results[-1]["accuracy"]:.4f} | F1={results[-1]["f1_score"]:.4f}')

## 11. Model Evaluation (Confusion Matrix & Classification Report)

In [ ]:
best_name_preview = max(results, key=lambda r: r['f1_score'])['model']
y_pred_preview = fitted[best_name_preview].predict(X_test)
print(f'Detailed report for: {best_name_preview}')
print(confusion_matrix(y_test, y_pred_preview, labels=['Low','Moderate','High']))
print(classification_report(y_test, y_pred_preview, zero_division=0))

## 12. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('f1_score', ascending=False).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(results_df)); width = 0.2
for i, m in enumerate(['accuracy','precision','recall','f1_score']):
    ax.bar(x + i*width, results_df[m], width, label=m)
ax.set_xticks(x + width*1.5); ax.set_xticklabels(results_df['model'], rotation=20, ha='right')
ax.set_title('Model Performance Comparison'); ax.legend()
plt.show()

## 13. Hyperparameter Tuning (GridSearchCV)

We tune the top-performing model from the comparison above. Below is a generic example for Random Forest / SVM — adapt the grid to whichever model scored highest.

In [ ]:
top_model_name = results_df.iloc[0]['model']
print('Tuning:', top_model_name)

param_grids = {
    'Random Forest': {'classifier__n_estimators':[100,200,300], 'classifier__max_depth':[None,8,12,16], 'classifier__min_samples_split':[2,5,10]},
    'SVM': {'classifier__C':[0.1,1,10], 'classifier__kernel':['rbf','linear'], 'classifier__gamma':['scale','auto']},
    'Decision Tree': {'classifier__max_depth':[4,8,12,None], 'classifier__min_samples_split':[2,5,10]},
    'KNN': {'classifier__n_neighbors':[3,5,7,9,11], 'classifier__weights':['uniform','distance']},
    'Logistic Regression': {'classifier__C':[0.1,1,10], 'classifier__solver':['lbfgs','liblinear']},
}

base_pipe = Pipeline([('preprocessor', preprocessor), ('classifier', models[top_model_name])])
search = GridSearchCV(base_pipe, param_grid=param_grids[top_model_name], cv=skf, scoring='f1_macro', n_jobs=-1)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)
print('Best CV F1 (macro):', search.best_score_)
best_pipeline = search.best_estimator_

## 14. Final Model — Evaluation on Test Set

In [ ]:
y_pred_final = best_pipeline.predict(X_test)
print('Final Test Accuracy:', accuracy_score(y_test, y_pred_final))
print('Final Test F1 (macro):', f1_score(y_test, y_pred_final, average='macro', zero_division=0))
print(confusion_matrix(y_test, y_pred_final, labels=['Low','Moderate','High']))
print(classification_report(y_test, y_pred_final, zero_division=0))

## 15. Save Final Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best_pipeline, '../models/burning_feet_model.pkl')
print('Model saved to ../models/burning_feet_model.pkl')

## 16. Feature Importance

For tree-based models we use the native `feature_importances_`. For other models (e.g. SVM, Logistic Regression) we fall back to **permutation importance**, which works for any fitted estimator.

In [ ]:
from sklearn.inspection import permutation_importance

clf = best_pipeline.named_steps['classifier']
if hasattr(clf, 'feature_importances_'):
    ohe = best_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
    cat_names = list(ohe.get_feature_names_out(categorical_cols))
    all_names = numeric_cols + cat_names + binary_cols
    fi = pd.DataFrame({'feature': all_names, 'importance': clf.feature_importances_}).sort_values('importance', ascending=False)
else:
    result = permutation_importance(best_pipeline, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='f1_macro', n_jobs=-1)
    fi = pd.DataFrame({'feature': list(X_test.columns), 'importance': result.importances_mean}).sort_values('importance', ascending=False)

fi.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
top_fi = fi.head(12)
ax.barh(top_fi['feature'][::-1], top_fi['importance'][::-1], color='teal')
ax.set_title(f'Feature Importance ({top_model_name})')
plt.show()

## 17. Conclusion

- Multiple classification algorithms (Logistic Regression, Decision Tree, Random Forest, KNN, SVM)
  were trained and compared using stratified 5-fold cross-validation and a held-out test set.
- The best-performing model was selected based on **macro F1-score** on the test set (not assumed
  in advance).
- Hyperparameters of the winning model were tuned using `GridSearchCV`.
- The final pipeline (preprocessing + model) was saved with `joblib` for reuse in the Streamlit
  application.
- Feature importance / permutation importance shows which inputs most influenced the model's
  predictions — this reflects **model behavior on synthetic data**, not medical causation.

**Reminder:** This entire project is trained on a synthetic/demonstration dataset for academic
purposes. It is not a validated diagnostic tool and must not be used for real medical decisions.